# 05 · 95th-Percentile Days-to-Trade (DTT) — Operational Replicability


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os, warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

PROJECT_DIR = r'C:\Users\name\Documents\Bachelor Thesis\Empirical Analysis'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')

# --- Same-window counterfactual --------------------------------------------
orig = pd.read_parquet(os.path.join(DATA_DIR, 'constituents_original_today.parquet'))
pf   = pd.read_parquet(os.path.join(DATA_DIR, 'constituents_proforma.parquet'))

orig = orig[orig['FF_MktCap'].notna()].copy()
pf   = pf[pf['FF_MktCap'].notna()].copy()
orig['w'] = orig['FF_MktCap'] / orig['FF_MktCap'].sum()
pf['w']   = pf['FF_MktCap']  / pf['FF_MktCap'].sum()

# --- Robust-window CLOSE / VOLUME (main + gap) ----------------------------
def _load_robust(kind):
    main_fp = os.path.join(DATA_DIR, f'{kind}_robust.parquet')
    gap_fp  = os.path.join(DATA_DIR, f'{kind}_robust_gap.parquet')
    frames = []
    if os.path.exists(main_fp):
        frames.append(pd.read_parquet(main_fp))
    if os.path.exists(gap_fp):
        frames.append(pd.read_parquet(gap_fp))
    df = pd.concat(frames, axis=1) if frames else pd.DataFrame()
    return df.loc[:, ~df.columns.duplicated()]

df_close  = _load_robust('prices')
df_volume = _load_robust('volume')
print(f'Original-today : {len(orig):,} stocks    Pro-forma-today : {len(pf):,} stocks')
print(f'Robust CLOSE : {df_close.shape}   VOLUME : {df_volume.shape}')


## 1  Parameters and ADTV computation

In [ ]:
# ── ADTV per stock on the 2025-26 window (ONE computation) ──────────────────
FUND_AUM = 1e12    # ¥ 1 trillion (representative large-scale passive fund)
ALPHA    = 0.20    # 20% participation rate (standard institutional assumption)

traded_value = df_close * df_volume
adtv         = traded_value.mean(axis=0)
adtv.name    = 'ADTV'

orig = orig.merge(pd.DataFrame({'RIC': adtv.index, 'ADTV': adtv.values}),
                  on='RIC', how='left')
pf   = pf.merge(pd.DataFrame({'RIC': adtv.index, 'ADTV': adtv.values}),
                on='RIC', how='left')

print(f'ADTV (2025-26) computed for {adtv.notna().sum():,} stocks')
print(f'  Median ADTV  Original-today  : ¥{orig["ADTV"].median():,.0f}')
print(f'  Median ADTV  Pro-forma-today : ¥{pf["ADTV"].median():,.0f}')


## 2  Compute DTT per stock and find 95th percentile

In [ ]:
# ── DTT per stock + P95 ──────────────────────────────────────────────────────
# DTT_i = (Fund_AUM * w_i) / (alpha * ADTV_i)
orig['DTT'] = (FUND_AUM * orig['w']) / (ALPHA * orig['ADTV'])
pf['DTT']   = (FUND_AUM * pf['w'])   / (ALPHA * pf['ADTV'])

orig['DTT'] = orig['DTT'].replace([np.inf, -np.inf], np.nan)
pf['DTT']   = pf['DTT'].replace([np.inf, -np.inf], np.nan)

p95_orig = orig['DTT'].quantile(0.95)
p95_pf   = pf['DTT'].quantile(0.95)

print('=' * 55)
print(f'  Fund AUM   : ¥{FUND_AUM:,.0f}')
print(f'  Alpha      : {ALPHA:.0%}')
print(f'  P95 DTT  Original-today  : {p95_orig:.2f} days')
print(f'  P95 DTT  Pro-forma-today : {p95_pf:.2f} days')
if p95_orig and not np.isnan(p95_orig):
    print(f'  Change (PF - Orig)       : {p95_pf - p95_orig:+.2f} days'
          f'  ({(p95_pf - p95_orig)/p95_orig*100:+.1f}%)')
print('=' * 55)

bottleneck_orig = orig[orig['DTT'] >= p95_orig].sort_values('DTT', ascending=False)
bottleneck_pf   = pf[pf['DTT']   >= p95_pf].sort_values('DTT', ascending=False)
print(f'\nBottleneck stocks (DTT ≥ P95):')
print(f'  Original-today  : {len(bottleneck_orig):,} stocks   max DTT = {orig["DTT"].max():.1f} days')
print(f'  Pro-forma-today : {len(bottleneck_pf):,} stocks   max DTT = {pf["DTT"].max():.1f} days')


## 2b  Significance tests aligned with the P95 headline

The headline DTT statistic in this thesis is the 95th percentile, because
operational replicability is a tail-risk problem: the binding constraint
is the slowest stocks to liquidate, not the median. Mean-based or
rank-based two-sample tests (Welch, Mann-Whitney U) speak to the centre
of the distribution and are therefore not the right inference for a
P95 claim. The two tests below target the 95th percentile directly.

**Primary — bootstrap CI for ΔP95.** Resample each index independently
B = 10,000 times, recompute the difference in P95 each time, report the
point estimate, the 95% percentile interval, and the one-sided
bootstrap p-value (H₁: P95ₒ > P95ₚ). This is non-parametric and
robust to the right-skew of DTT. Theoretical foundation:
Ghosh (1971), Falk & Reiss (1989).

**Robustness — quantile regression at τ = 0.95.** Regress DTT on a binary
group dummy (1 = pro-forma) at the 0.95 quantile via statsmodels.QuantReg.
The coefficient β̂ is the conditional P95 shift; its sign, SE, and
one-sided p-value test the same null in a parametric, regression-style
form familiar to finance readers.


In [ ]:
# ── P95-aligned significance tests on DTT ────────────────────────────
# Two methods, both targeting the 95th percentile (the headline statistic):
#   Primary    : bootstrap CI for ΔP95 (orig - pf), B = 10,000, one-sided.
#   Robustness : quantile regression at τ = 0.95 with binary group dummy.
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

dtt_orig = orig['DTT'].dropna().astype(float).to_numpy()
dtt_pf   = pf['DTT'].dropna().astype(float).to_numpy()

p95_orig = float(np.percentile(dtt_orig, 95))
p95_pf   = float(np.percentile(dtt_pf,   95))
delta_p95_obs = p95_orig - p95_pf

print(f'N orig (non-NaN DTT): {len(dtt_orig):,}')
print(f'N pf   (non-NaN DTT): {len(dtt_pf):,}')
print(f'P95 orig: {p95_orig:.3f}   P95 pf: {p95_pf:.3f}')
print(f'Observed ΔP95 (orig - pf): {delta_p95_obs:+.3f}')
if p95_orig > 0:
    print(f'Relative reduction: {(delta_p95_obs / p95_orig):.2%}')
print()

# --- Primary: bootstrap CI for ΔP95 -----------------------------------------
B = 10_000
rng = np.random.default_rng(seed=20260425)

n_o, n_p = len(dtt_orig), len(dtt_pf)
boot_deltas = np.empty(B, dtype=float)
for b in range(B):
    s_o = dtt_orig[rng.integers(0, n_o, size=n_o)]
    s_p = dtt_pf  [rng.integers(0, n_p, size=n_p)]
    boot_deltas[b] = np.percentile(s_o, 95) - np.percentile(s_p, 95)

ci_lo, ci_hi = np.percentile(boot_deltas, [2.5, 97.5])
# One-sided p (H1: orig > pf): share of resamples with Δ ≤ 0
boot_p_one = float(np.mean(boot_deltas <= 0))

print('Primary — Bootstrap CI for ΔP95 (B = 10,000, one-sided H₁: orig > pf):')
print(f'  Point estimate ΔP95 : {delta_p95_obs:+.3f}')
print(f'  Bootstrap mean ΔP95 : {boot_deltas.mean():+.3f}')
print(f'  95% percentile CI    : [{ci_lo:+.3f}, {ci_hi:+.3f}]')
print(f'  One-sided bootstrap p: {boot_p_one:.4e}')
reject_boot = (ci_lo > 0)
print(f'  CI excludes 0?       : {reject_boot}  '
      f'→ {"reject H0" if reject_boot else "fail to reject H0"}')
print()

# --- Robustness: quantile regression at τ = 0.95 ----------------------------
qr_df = pd.DataFrame({
    'DTT'  : np.concatenate([dtt_orig, dtt_pf]),
    'is_pf': np.concatenate([np.zeros(n_o), np.ones(n_p)]),
})

qr_model = smf.quantreg('DTT ~ is_pf', data=qr_df)
qr_res   = qr_model.fit(q=0.95)
beta   = float(qr_res.params['is_pf'])
se     = float(qr_res.bse['is_pf'])
tval   = float(qr_res.tvalues['is_pf'])
p_two  = float(qr_res.pvalues['is_pf'])
# One-sided: H1 says pf shifts the 0.95-quantile downward (β < 0)
if beta < 0:
    p_one = p_two / 2.0
else:
    p_one = 1.0 - p_two / 2.0

print('Robustness — Quantile regression DTT ~ is_pf at τ = 0.95:')
print(f'  β̂ (pf shift in conditional P95): {beta:+.3f}')
print(f'  SE                              : {se:.3f}')
print(f'  t                               : {tval:+.3f}')
print(f'  Two-sided p                     : {p_two:.4e}')
print(f'  One-sided p (H₁: β < 0)        : {p_one:.4e}')
print()

# --- Joint verdict -----------------------------------------------------------
if reject_boot and p_one < 0.05:
    print('Both tests reject H0 at α = 0.05 — the pro-forma index has a '
          'significantly lower 95th-percentile DTT than the original index.')
elif reject_boot or p_one < 0.05:
    print(f'Mixed result: bootstrap reject={reject_boot}, QuantReg p={p_one:.3g}. '
          'Prefer the bootstrap given the right-skew of DTT.')
else:
    print('Neither test rejects H0 at α = 0.05 — no significant P95 difference.')


## 3  Sensitivity analysis: vary Fund AUM and alpha

In [ ]:
# ── Sensitivity across AUM × alpha grid ──────────────────────────────────────
aum_grid   = [5e11, 1e12, 2e12, 5e12]
alpha_grid = [0.10, 0.20, 0.30]

rows = []
for aum in aum_grid:
    for alpha in alpha_grid:
        p95_o = ((aum * orig['w']) / (alpha * orig['ADTV'])).replace(
            [np.inf, -np.inf], np.nan).quantile(0.95)
        p95_p = ((aum * pf['w'])   / (alpha * pf['ADTV'])).replace(
            [np.inf, -np.inf], np.nan).quantile(0.95)
        rows.append({
            'Fund AUM (¥ trillion)': aum / 1e12,
            'Alpha': f'{alpha:.0%}',
            'P95 DTT Original-today':  round(p95_o, 2),
            'P95 DTT Pro-forma-today': round(p95_p, 2),
            'Change (days)': round(p95_p - p95_o, 2),
        })

sensitivity = pd.DataFrame(rows)
print(sensitivity.to_string(index=False))


## 4  Visualisation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# A  DTT distribution (log)
ax = axes[0, 0]
ax.hist(np.log10(orig['DTT'].dropna().clip(lower=1e-4)), bins=60,
        alpha=0.6, label='Original-today',  color='steelblue')
ax.hist(np.log10(pf['DTT'].dropna().clip(lower=1e-4)), bins=60,
        alpha=0.6, label='Pro-forma-today', color='darkorange')
ax.axvline(np.log10(max(p95_orig, 1e-4)), color='steelblue',  ls='--', lw=1.5, label=f'P95 O = {p95_orig:.1f}d')
ax.axvline(np.log10(max(p95_pf,   1e-4)), color='darkorange', ls='--', lw=1.5, label=f'P95 P = {p95_pf:.1f}d')
ax.set_xlabel('log₁₀(DTT in days)'); ax.set_ylabel('Number of stocks')
ax.set_title('A  DTT Distribution (log scale)')
ax.legend(fontsize=8)

# B  P95 DTT bar
ax = axes[0, 1]
bars = ax.bar(['Original-today', 'Pro-forma-today'],
              [p95_orig, p95_pf],
              color=['steelblue', 'darkorange'], width=0.4, edgecolor='black')
ax.bar_label(bars, fmt='{:.2f}d', padding=3, fontsize=10)
ax.set_ylabel('95th-percentile DTT (days)')
ax.set_title('B  95th-Percentile Days-to-Trade')

# C  Cumulative DTT
ax = axes[1, 0]
for label, df_p, color in [('Original-today',  orig, 'steelblue'),
                           ('Pro-forma-today', pf,   'darkorange')]:
    sorted_dtt = np.sort(df_p['DTT'].dropna().values)
    if len(sorted_dtt):
        ax.plot(sorted_dtt, np.arange(1, len(sorted_dtt) + 1) / len(sorted_dtt),
                label=label, color=color)
ax.axhline(0.95, color='gray', ls=':', lw=1)
ax.set_xscale('log')
ax.set_xlabel('DTT (days, log scale)'); ax.set_ylabel('Cumulative fraction of stocks')
ax.set_title('C  Cumulative DTT Distribution')
ax.legend()

# D  Sensitivity heat-map
ax = axes[1, 1]
piv = sensitivity.pivot(index='Fund AUM (¥ trillion)',
                        columns='Alpha',
                        values='Change (days)')
sns.heatmap(piv, ax=ax, annot=True, fmt='.1f', cmap='RdBu_r', center=0,
            cbar_kws={'label': 'ΔP95 DTT (Pro-forma − Original, days)'})
ax.set_title('D  DTT Change Sensitivity (days)')

plt.suptitle('Days-to-Trade — TOPIX Reform (same-window counterfactual, 2025-26)', fontsize=12)
plt.tight_layout()
plt.savefig('dtt_analysis.png', bbox_inches='tight')
plt.show()
print('Figure saved: dtt_analysis.png')


In [ ]:
dtt_orig_export = orig[['RIC'] + [c for c in ['Name'] if c in orig.columns] + ['w', 'ADTV', 'DTT']].copy()
dtt_pf_export   = pf[  ['RIC'] + [c for c in ['Name'] if c in pf.columns]   + ['w', 'ADTV', 'DTT']].copy()
dtt_orig_export.to_csv('dtt_original.csv', index=False)
dtt_pf_export.to_csv('dtt_proforma.csv', index=False)
sensitivity.to_csv('dtt_sensitivity.csv', index=False)

print('Saved: dtt_original.csv, dtt_proforma.csv, dtt_sensitivity.csv')
print(f'\nBase case (AUM=¥1T, alpha=20%):')
print(f'  P95 DTT Original-today  : {p95_orig:.2f} days')
print(f'  P95 DTT Pro-forma-today : {p95_pf:.2f} days')
print(f'  Change                  : {p95_pf - p95_orig:+.2f} days  ({(p95_pf-p95_orig)/p95_orig*100:+.1f}%)')
